# Privacy-Preserving Federated Learning: Comparison Analysis

Modes included:
- **baseline** — No privacy (FedAvg)
- **he_tenseal** — HE-TenSEAL (CKKS approximate homomorphic encryption)
- **he_concrete_tfhe** — HE-Concrete TFHE (exact boolean-circuit FHE)
- **zkp** — Zero-Knowledge Proofs (gnark Groth16 zk-SNARK, all layers)
- **zkp_sampled** — ZKP with sampled-layer proofs
- **dp** — Differential Privacy (Gaussian noise, Opacus)
- **he_tenseal_zkp** — Hybrid: HE-TenSEAL + ZKP integrity proofs
- **he_concrete_tfhe_zkp** — Hybrid: HE-Concrete TFHE + ZKP integrity proofs
- **he_tenseal_zkp_dp** — Hybrid: HE-TenSEAL + ZKP + DP
- **he_concrete_tfhe_zkp_dp** — Hybrid: HE-Concrete TFHE + ZKP + DP

**Datasets:** Healthcare · Credit Card · Stock · MNIST · CIFAR-10

All results are loaded from `results/<dataset>/comparison_report.json` which is updated automatically after each `compare.py` run.


In [ ]:
# Helper: load cleaned comparison report
import json, os
from pathlib import Path

def load_report(dataset):
    p = Path(f'results/{dataset}/comparison_report.json')
    if not p.exists():
        print('Report missing:', p)
        return None
    with open(p, 'r') as f:
        return json.load(f)

# Example: report = load_report('creditcard')
# print(len(report) if isinstance(report, list) else report.keys())

# Privacy-Preserving Federated Learning: Comprehensive Comparison Analysis

This notebook provides a detailed comparison of privacy-preserving methods in federated learning:
1. **Baseline** — No privacy (plain FedAvg)
2. **DP** — Differential Privacy (Gaussian noise, Opacus). Formal epsilon-delta privacy guarantee; small accuracy degradation.
3. **ZKP** — Zero-Knowledge Proofs (gnark Groth16 zk-SNARK). Proves gradient integrity without revealing values.
4. **ZKP-Sampled** — ZKP with sampled-layer proofs for lower prover cost per round.
5. **HE-TenSEAL** — CKKS Approximate Homomorphic Encryption (TenSEAL). Packs entire weight vectors into CKKS ciphertexts using SIMD-style batching (BatchCrypt approach). Approximate arithmetic; small decryption noise around 1e-9.
6. **HE-Concrete-TFHE** — Exact FHE with quantized boolean circuits (Concrete ML). Up to 32x larger ciphertexts than CKKS, exact arithmetic.
7. **HE-TenSEAL + ZKP** — Hybrid: CKKS encryption for confidentiality + ZKP for integrity verification.
8. **HE-Concrete-TFHE + ZKP** — Hybrid: exact FHE confidentiality + ZKP integrity.
9. **HE-TenSEAL + ZKP + DP** — Hybrid confidentiality + integrity + formal differential privacy.
10. **HE-Concrete-TFHE + ZKP + DP** — Maximum-assurance hybrid with exact FHE, ZKP, and DP.

**Datasets:**
- **Healthcare** — Heart Disease Prediction (918 samples, 13 features, binary classification)
- **Credit Card** — Credit Card Fraud Detection (284,807 transactions, 0.17% fraud rate, binary; AUPRC is primary metric due to severe class imbalance)
- **Stock** — Stock Market Prediction (3,180 samples, 11 features)
- **MNIST** — Handwritten digit classification (10 classes, grayscale images)
- **CIFAR-10** — Natural image classification (10 classes, RGB images)

**Author:** TxCorpi0x | Last results: 2026-03-22


## 0. Theoretical Background (Cryptographic Context)

This section provides the formal mathematical and threat-model context for the 10 privacy modes benchmarked in this notebook:

- **baseline** — No privacy (FedAvg)
- **he_tenseal** — HE-TenSEAL (CKKS approximate homomorphic encryption)
- **he_concrete_tfhe** — HE-Concrete TFHE (exact boolean-circuit FHE)
- **zkp** — Zero-Knowledge Proofs (gnark Groth16 zk-SNARK, all layers)
- **zkp_sampled** — ZKP with sampled-layer proofs
- **dp** — Differential Privacy (Gaussian noise, Opacus)
- **he_tenseal_zkp** — Hybrid: HE-TenSEAL + ZKP integrity proofs
- **he_concrete_tfhe_zkp** — Hybrid: HE-Concrete TFHE + ZKP integrity proofs
- **he_tenseal_zkp_dp** — Hybrid: HE-TenSEAL + ZKP + DP
- **he_concrete_tfhe_zkp_dp** — Hybrid: HE-Concrete TFHE + ZKP + DP

**Datasets:** Healthcare · Credit Card · Stock · MNIST · CIFAR-10

---

### 1. Threat Model and Security Objectives

The FL setting in this notebook assumes two primary adversary classes:

1. **Honest-but-curious aggregator**: follows protocol but attempts to infer private training data from client updates.
2. **Malicious (Byzantine) client**: can submit manipulated gradients to poison or backdoor global training.

Corresponding cryptographic goals:

1. **Confidentiality**: hide plaintext gradients/weights from server and external observers.
2. **Integrity / correctness**: prove client updates satisfy protocol constraints.
3. **Formal statistical privacy**: bound individual contribution leakage with epsilon-delta guarantees.

The evaluated methods map to these goals as follows:

1. **Baseline** — No privacy (plain FedAvg)
2. **DP** — Differential Privacy (Gaussian noise, Opacus). Formal epsilon-delta privacy guarantee; small accuracy degradation.
3. **ZKP** — Zero-Knowledge Proofs (gnark Groth16 zk-SNARK). Proves gradient integrity without revealing values.
4. **ZKP-Sampled** — ZKP with sampled-layer proofs for lower prover cost per round.
5. **HE-TenSEAL** — CKKS approximate homomorphic encryption for encrypted aggregation.
6. **HE-Concrete-TFHE** — exact FHE aggregation via boolean-circuit cryptography.
7. **HE-TenSEAL + ZKP** — Hybrid: CKKS encryption for confidentiality + ZKP for integrity verification.
8. **HE-Concrete-TFHE + ZKP** — Hybrid: exact FHE confidentiality + ZKP integrity.
9. **HE-TenSEAL + ZKP + DP** — Hybrid confidentiality + integrity + formal differential privacy.
10. **HE-Concrete-TFHE + ZKP + DP** — Maximum-assurance hybrid with exact FHE, ZKP, and DP.

---

### 1A. Threat Landscape in Federated Learning

The table below maps major FL attack classes to corresponding defenses benchmarked in this work.

| Attack Class | Adversary | Mechanism | Primary Mitigation in This Framework |
|--------------|-----------|-----------|--------------------------------------|
| **Gradient Inversion / Reconstruction** | Honest-but-curious server | Adversary optimises dummy inputs to match received gradients, recovering training samples (Zhu et al. 2019; Geiping et al. 2021) | HE (server sees only ciphertexts); DP (gradients randomised before upload) |
| **Membership Inference** | Server or external | Adversary distinguishes whether a target record was in a client's training set by probing the global model (Shokri et al., IEEE S&P 2017) | DP (noise obfuscates individual contributions); HE (model gradient opaque on server) |
| **Byzantine / Poisoning** | Malicious client | A colluding client submits crafted gradients designed to skew the global model or embed a backdoor — persists even under HE, since the encrypted update can be maliciously formed | ZKP (proves updates satisfy arithmetic validity constraints e.g., bounded L2 norm, correct local SGD step) |

The **honest-but-curious (semi-honest)** adversary model assumes the server executes the protocol faithfully but attempts to learn private information from messages it observes. HE is the canonical defence here. The **malicious** adversary model assumes the client may deviate from the protocol specification — ZKP provides cryptographically verifiable integrity in this model.

---

### 2. CKKS (TenSEAL) — Approximate Homomorphic Encryption

#### 2.1 Mathematical Foundation

CKKS (Cheon, Kim, Kim, Song 2017) is an RLWE-based leveled homomorphic encryption scheme over the cyclotomic ring

$$
R_q = \mathbb{Z}_q[X]/(X^N + 1), \quad N = 2^k
$$

with plaintext space embedded into $\mathbb{C}^{N/2}$. Ciphertexts are polynomial pairs $(c_0, c_1) \in R_q^2$ such that decryption yields:

$$
c_0 + c_1 s = m + e \pmod q,
$$

where $s$ is the secret key polynomial and $e$ is sampled from a discrete Gaussian $\chi$ with small standard deviation $\sigma_e$. This problem is believed computationally hard for both classical and quantum adversaries (lattice-basis reduction algorithms achieve sub-exponential but not polynomial time even with quantum resources).

#### 2.2 SIMD Batching for Federated Learning

A single CKKS ciphertext encodes a complex vector of dimension $N/2$ via the canonical embedding (a ring isomorphism). For federated learning, this enables SIMD-style batching: an entire neural network weight array is packed into one ciphertext, and homomorphic addition applies element-wise to all weights simultaneously:

$$
\text{Enc}(\mathbf{w}_1) \oplus \text{Enc}(\mathbf{w}_2) = \text{Enc}(\mathbf{w}_1 + \mathbf{w}_2)
$$

BatchCrypt (Zhang et al., USENIX ATC 2020) formalises this for cross-silo FL: gradient aggregation reduces to a single homomorphic addition per client per layer — no fan-in multiplication, no bootstrapping. The dominant cost shifts to network transmission (~657 KB/round in this work), not computation.

#### 2.3 Approximate Arithmetic

CKKS trades exact correctness for computational efficiency: every rescaling introduces a bounded noise term $\varepsilon \approx 2^{-\ell}$ (where $\ell$ is the remaining ciphertext level). For gradient aggregation (addition only — no multiplication required), the accumulated noise satisfies $\varepsilon < 10^{-9}$, well below the numerical precision of IEEE 754 double-precision training. In practice, decrypted gradients are numerically indistinguishable from their plaintext counterparts for practical learning purposes.

TenSEAL (Benaissa et al. 2021) is used here: a Python interface over Microsoft SEAL implementing `ckks_vector`, which packs a flat 1-D weight array into a single ciphertext. Server-side FedAvg averaging is performed directly on ciphertext objects — the aggregator never accesses plaintext gradients.

---

### 3. TFHE — Torus Fully Homomorphic Encryption

#### 3.1 Mathematical Foundation

TFHE (Chillotti, Gama, Georgieva & Izabachene, Journal of Cryptology 2020) is based on the Torus LWE (TLWE) problem. A TLWE ciphertext encrypting message $\mu \in \mathbb{T} = \mathbb{R}/\mathbb{Z}$ has the form $(a_1, \ldots, a_n, b) \in \mathbb{T}^{n+1}$ where:

$$
b = \mathbf{s} \cdot \mathbf{a} + \mu + e \pmod{1}, \quad e \sim \mathcal{N}(0, \sigma_e^2)
$$

Secret key $\mathbf{s} \in \{0,1\}^n$ and Gaussian noise $e$ are small relative to the modulus; security reduces to the hardness of lattice problems (LWE with $n \geq 1024$ achieves >=128-bit classical and quantum security). Concrete-ML targets 128-bit security by default via $n = 1024$, $\sigma_e = 2^{-40}$.

#### 3.2 Programmable Bootstrapping

Unlike CKKS, TFHE evaluates arbitrary boolean circuits gate-by-gate. Accumulated noise is fully reset via a programmable bootstrapping step within each gate — a homomorphic evaluation of a look-up table (LUT) over TGLWE ciphertexts. This yields exact (zero-error) computation on quantized integers, at the cost of one bootstrapping operation per gate (~10–100 ms/gate on CPU), which is the dominant TFHE performance bottleneck.

#### 3.3 Concrete-ML Integration

Zama's Concrete-ML compiles standard PyTorch / scikit-learn models to TFHE circuits via quantisation-aware training:
1. Quantise model weights and activations to $b$-bit integers (typically $b \in \{4, 6, 8\}$).
2. Compile the quantised computation graph to a Rank-1 Constraint System (R1CS) / boolean circuit.
3. Encrypt each quantised parameter in its own LWE/GLWE ciphertext.

Each scalar weight requires its own ciphertext, yielding an upload overhead of $N_\text{params} \times b_\text{bits} \times C_\text{ct}$ where $C_\text{ct} \approx 33\,\text{KB}$. For the neural network architectures evaluated here this results in 26.5 MB per round — a 1,650x increase vs. baseline. In WAN deployments, this gap is expected to dominate total training time.

---

### 4. Differential Privacy — Formal (epsilon, delta)-Guarantee

#### 4.1 Definition

A randomised algorithm $\mathcal{M}: \mathcal{D} \to \mathcal{R}$ satisfies $(\varepsilon, \delta)$-differential privacy (Dwork & Roth, 2014) if, for any two adjacent datasets $D, D' \in \mathcal{D}$ differing by exactly one individual's record, and all measurable $S \subseteq \mathcal{R}$:

$$
\Pr[\mathcal{M}(D) \in S] \leq e^{\varepsilon} \cdot \Pr[\mathcal{M}(D') \in S] + \delta
$$

Here, $\varepsilon$ (the privacy budget) controls the worst-case distinguishability: $\varepsilon = 0$ is perfect privacy, $\varepsilon \ll 1$ provides strong indistinguishability. $\delta$ allows a $\delta$-probability exception (typically $\delta \ll 1/|D|$).

#### 4.2 DP-SGD Mechanism

Abadi et al. (ACM CCS 2016) apply DP to stochastic gradient descent via three operations applied per mini-batch $B$:

1. Per-sample gradient clipping: $\bar{g}_i = g_i / \max\!\left(1, \|g_i\|_2 / C\right)$, bounding $\ell_2$ sensitivity to $\Delta f = C$.
2. Gaussian noise addition: $\tilde{g} = \frac{1}{|B|}\left(\sum_{i \in B} \bar{g}_i + \mathcal{N}(0, \sigma^2 C^2 \mathbf{I})\right)$
3. Privacy accounting: cumulative privacy cost across rounds tracked via the moments accountant or Renyi Differential Privacy (Mironov 2017), enabling tight composition.

Opacus (Meta) implements DP-SGD as a drop-in replacement for standard PyTorch optimisers. The observed ~19% overhead vs. baseline in this work originates primarily from per-sample gradient materialisation (PyTorch must retain per-example gradients before clipping), not from noise sampling, which is an $O(d)$ operation.

#### 4.3 Threat Model and Limitations

DP provides a statistical indistinguishability guarantee: even an computationally unbounded aggregator cannot distinguish individual contributions beyond the $e^\varepsilon$ bound. However:
- The aggregator observes (noisy) gradients — no confidentiality from the server's perspective.
- DP does not prevent Byzantine poisoning (a malicious client can add adversarial gradients that survive within the clipping budget).
- Composition across rounds accumulates privacy cost; the total budget $\varepsilon_\text{total}$ must be carefully managed in long-running FL deployments.

---

### 5. Zero-Knowledge Proofs — Groth16 zk-SNARK

#### 5.1 Cryptographic Properties

A zero-knowledge proof system $(\text{Gen}, \text{Prove}, \text{Verify})$ for a relation $R \subseteq \mathcal{X} \times \mathcal{W}$ satisfies three properties:
- Completeness: $\forall (x, w) \in R,\; \Pr[\text{Verify}(x, \text{Prove}(x, w)) = 1] = 1$.
- Soundness: for any $x \notin L_R$ (the language of $R$) and any cheating prover $P^*$, $\Pr[\text{Verify}(x, P^*(x)) = 1] \leq 2^{-\lambda}$ (negligible).
- Zero-knowledge: the verifier learns nothing beyond the truth of $(x, w) \in R$; the proof transcript $\pi$ is computationally indistinguishable from a simulator's output that does not know $w$.

#### 5.2 Groth16 Construction

Groth16 (EUROCRYPT 2016) is the dominant zk-SNARK protocol for arithmetic circuits over a prime field $\mathbb{F}_p$. The construction proceeds via:

1. R1CS: the computation is expressed as a Rank-1 Constraint System — a system of quadratic equations $(A \cdot z) \circ (B \cdot z) = C \cdot z$ over witness vector $z$.
2. QAP: the R1CS is transformed to a Quadratic Arithmetic Program via Lagrange interpolation — soundness reduces to the q-Power Knowledge of Exponent assumption over bilinear groups.
3. Pairing-based proof: the prover commits to witness polynomials at a structured reference string (SRS) encoded in elliptic curve group elements (BN254 or BLS12-381). The proof $\pi = (\alpha, \beta, \gamma) \in \mathbb{G}_1^2 \times \mathbb{G}_2$ consists of three curve points (~192 bytes), independent of circuit size.

Verification requires $O(1)$ pairing evaluations — constant time regardless of witness size or circuit depth.

#### 5.3 Application to Federated Learning

In this work, each client generates a Groth16 proof that its gradient update satisfies a validity constraint (e.g., $\|\nabla w\|_2 \leq C$, correct SGD computation). The server verifies proofs using only the public verification key before aggregation — it never needs the raw gradient. This provides:

- Byzantine robustness: a malicious client cannot submit a constraint-violating gradient and pass verification, as soundness holds under the discrete-log hardness of the elliptic curve group.
- Auditability: proof transcripts provide a non-repudiable audit trail — each client's claim of computational correctness is independently verifiable by any third party holding the verification key.

gnark (Ethereum Foundation, Go) implements Groth16 over BN254. Proof generation scales as $O(N_\text{gates} \log N_\text{gates})$ via Number Theoretic Transform (NTT)-based multi-scalar multiplication — explaining the ~10 s/round cost, driven by model size (number of arithmetic gates), not dataset size.

ZKP-Sampled generates proofs for a randomly selected subset of layers per round. This Randomised Probabilistic Proof (RPP) approach reduces per-round cost proportionally while maintaining statistical coverage across training — each layer is proven with high probability $1 - (1-p)^T$ after $T$ rounds with sampling probability $p$.

---

### 6. Hybrid Approaches: Confidentiality ⊕ Integrity (and DP-Augmented Variants)

The HE + ZKP family combines two orthogonally complementary mechanisms, and can be extended with DP for formal statistical privacy:

| Property | HE (CKKS / TFHE) | ZKP (Groth16) | HE + ZKP | HE + ZKP + DP |
|----------|------------------|---------------|----------|---------------|
| Gradient confidentiality | Yes (IND-CPA) | No | Yes | Yes |
| Byzantine robustness | No | Yes | Yes | Yes |
| Formal statistical privacy | Computational | Computational | Computational | High (epsilon-delta) |
| Passive adversary | Yes | Yes | Yes | Yes |
| Active adversary | No | Yes | Yes | Yes |
| Overhead (additive) | Low-Med | High | Highest | Highest+ |

For non-DP hybrid modes, HE-TenSEAL + ZKP and HE-Concrete-TFHE + ZKP provide a strong two-property security profile: cryptographic confidentiality plus verifiable integrity.

When DP is added (HE + ZKP + DP), the system also gains formal statistical privacy; these DP-augmented hybrids are therefore the strongest overall assurance tier in this notebook.

---

### 7. The Privacy-Utility-Efficiency Trilemma

Privacy-preserving methods in FL exhibit a fundamental three-way tension:

| Property | Baseline | DP | HE-CKKS | HE-TFHE | ZKP | HE+ZKP | HE+ZKP+DP |
|----------|----------|----|---------|---------|-----|--------|-----------|
| Confidentiality | No | Partial (noisy gradients visible) | Very High (IND-CPA) | Very High (IND-CPA, exact) | No | Very High | Very High |
| Integrity | No | No | No | No | Very High (sound) | Very High | Very High |
| Formal statistical privacy | No | High (epsilon-delta) | Computational | Computational | Computational | Computational | High (epsilon-delta) |
| Model utility (observed) | Very High | High | Very High | Very High | Very High | High-Very High | High |
| Compute efficiency | Very High | High | High | Low | Low | Low | Lowest |
| Communication overhead | 1x | 1x | ~41x | ~1,650x | 1x | ~41x to ~1,650x | Similar to HE+ZKP |

No single method dominates all dimensions. In this notebook, HE+ZKP+DP occupies the highest-assurance corner, while DP or HE-TenSEAL can be preferable in resource-constrained deployments.

---

Key References:
- McMahan et al. (2017). Communication-Efficient Learning of Deep Networks from Decentralized Data. AISTATS.
- Cheon, Kim, Kim & Song (2017). Homomorphic Encryption for Arithmetic of Approximate Numbers. ASIACRYPT.
- Zhang et al. (2020). BatchCrypt: Efficient Homomorphic Encryption for Cross-Silo Federated Learning. USENIX ATC.
- Chillotti, Gama, Georgieva & Izabachene (2020). TFHE: Fast Bootstrapping over the Torus. Journal of Cryptology.
- Dwork & Roth (2014). The Algorithmic Foundations of Differential Privacy. Foundations and Trends in TCS.
- Abadi et al. (2016). Deep Learning with Differential Privacy. ACM CCS.
- Groth (2016). On the Size of Pairing-Based Non-interactive Arguments. EUROCRYPT.
- Zhu, Liu & Song (2019). Deep Leakage from Gradients. NeurIPS.
- Shokri et al. (2017). Membership Inference Attacks Against Machine Learning Models. IEEE S&P.
- Benaissa et al. (2021). TenSEAL: A Library for Encrypted Tensor Operations Using Homomorphic Encryption. ICLR PPML Workshop.
- Mironov (2017). Renyi Differential Privacy of the Gaussian Mechanism. IEEE CSF.

### 6A. Addendum: DP-Augmented Hybrid Modes

This section supersedes any earlier wording that identified only **HE + ZKP** as the strongest tier.
The notebook also evaluates DP-augmented hybrids, which provide defense-in-depth across all three dimensions: confidentiality, integrity, and formal statistical privacy.

- **HE-TenSEAL + ZKP + DP**: encrypted aggregation + proof-based integrity + epsilon-delta privacy.
- **HE-Concrete-TFHE + ZKP + DP**: exact encrypted aggregation + proof-based integrity + epsilon-delta privacy.

Conceptually, these modes extend HE+ZKP by adding client-side DP noise before upload, so threat coverage includes:
- Honest-but-curious server confidentiality via HE
- Byzantine client robustness via ZKP
- Membership and reconstruction resistance reinforcement via DP composition

In short, the strongest assurance tier in this notebook is **HE + ZKP + DP** (with higher runtime and communication cost than non-DP hybrids).

## 1. Setup and Imports

In [ ]:
# Standard libraries
import json
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Style configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("✅ All libraries imported successfully!")

## 2. Run Experiments

Configure the global parameters below, then execute the cell to run `compare.py` for every dataset the requested number of times.  
Each run produces a timestamped sub-directory under `results/<dataset>/` and updates `results/<dataset>/comparison_report.json` with the latest results.

In [ ]:
import sys
from pathlib import Path

# Canonical mode list (all supported privacy modes)
MODE_KEYS = [
    'baseline',
    'dp',
    'he_tenseal',
    'he_concrete_tfhe',
    'zkp',
    'zkp_sampled',
    'he_tenseal_zkp',
    'he_concrete_tfhe_zkp',
    'he_tenseal_zkp_dp',
    'he_concrete_tfhe_zkp_dp',
]
MODES = ','.join(MODE_KEYS)

# Canonical dataset list with report fallbacks
DATASET_SPECS = [
    {'run_key': 'healthcare', 'report_keys': ['healthcare'], 'label': 'Healthcare'},
    {'run_key': 'creditcard', 'report_keys': ['creditcard'], 'label': 'Credit Card'},
    {'run_key': 'stock', 'report_keys': ['stock'], 'label': 'Stock'},
    {'run_key': 'mnist', 'report_keys': ['mnist'], 'label': 'MNIST'},
    {'run_key': 'cifar10', 'report_keys': ['cifar10', 'cifar'], 'label': 'CIFAR-10'},
]

# -- Global experiment configuration -----------------------------------------
ROUNDS = 10      # FL rounds per experiment run
NUM_CLIENTS = 2  # number of federated clients
NUM_RUNS = 2     # how many times to repeat each dataset
DATASETS = [s['run_key'] for s in DATASET_SPECS]

# -- Print terminal commands --------------------------------------------------
# Running compare.py as a subprocess inside the notebook causes OOM crashes
# because the kernel + child process together exceed available RAM.
# Run the commands below in a SEPARATE terminal instead, then come back here
# to run the analysis cells (section 3 onward).

python = sys.executable
cwd = Path.cwd()

print('=' * 72)
print('  Run the following commands in a terminal (NOT in this notebook):')
print('=' * 72)
print(f'\n  cd {cwd}\n')

for dataset in DATASETS:
    for run_idx in range(NUM_RUNS):
        cmd = (
            f"{python} -W ignore compare.py"
            f" --dataset {dataset}"
            f" --rounds {ROUNDS}"
            f" --num-clients {NUM_CLIENTS}"
            f" --modes {MODES}"
        )
        label = f"# {dataset.upper()} run {run_idx + 1}/{NUM_RUNS}"
        print(f"  {label}")
        print(f"  {cmd}\n")

print('=' * 72)
print('  When all runs complete, execute the cells below to load and analyze')
print('  results from results/<dataset>/comparison_report.json')
print('=' * 72)

## 2. Load Results Data

In [ ]:
def load_comparison_results(report_keys, label):
    """Load comparison results for a dataset, trying fallback report keys."""
    report_path = None
    for key in report_keys:
        candidate = Path(f'results/{key}/comparison_report.json')
        if candidate.exists():
            report_path = candidate
            break

    if report_path is None:
        print(f"Report not found for {label}. Tried keys: {report_keys}")
        return None

    with open(report_path, 'r') as f:
        data = json.load(f)

    # Support both formats:
    #   - flat list  (current):  [{"mode": "baseline", ...}, ...]
    #   - dict       (legacy):   {"results": [{"mode": "baseline", ...}, ...]}
    if isinstance(data, list):
        items = data
        data = {'results': items}
    elif isinstance(data, dict) and 'results' in data and isinstance(data['results'], list):
        items = data['results']
    else:
        items = []

    modes_dict = {r.get('mode', 'unknown'): r for r in items if r.get('mode')}
    data['modes'] = modes_dict

    print(f"Loaded {label}: {len(modes_dict)} modes -- {sorted(modes_dict.keys())}")
    return data

# Rebuild dataset metadata if this cell is run independently
if 'DATASET_SPECS' not in globals():
    DATASET_SPECS = [
        {'run_key': 'healthcare', 'report_keys': ['healthcare'], 'label': 'Healthcare'},
        {'run_key': 'creditcard', 'report_keys': ['creditcard'], 'label': 'Credit Card'},
        {'run_key': 'stock', 'report_keys': ['stock'], 'label': 'Stock'},
        {'run_key': 'mnist', 'report_keys': ['mnist'], 'label': 'MNIST'},
        {'run_key': 'cifar10', 'report_keys': ['cifar10', 'cifar'], 'label': 'CIFAR-10'},
    ]

DATASET_KEYS = [s['run_key'] for s in DATASET_SPECS]
DATASET_LABELS = {s['run_key']: s['label'] for s in DATASET_SPECS}

results_by_dataset = {
    s['run_key']: load_comparison_results(s['report_keys'], s['label'])
    for s in DATASET_SPECS
}

# Keep backwards-compatible variables used later in the notebook
healthcare_results = results_by_dataset.get('healthcare')
creditcard_results = results_by_dataset.get('creditcard')
stock_results = results_by_dataset.get('stock')
mnist_results = results_by_dataset.get('mnist')
cifar10_results = results_by_dataset.get('cifar10')
cifar_results = cifar10_results  # legacy alias

# Display available modes
for spec in DATASET_SPECS:
    key = spec['run_key']
    data = results_by_dataset.get(key)
    if data and 'modes' in data:
        print(f"{spec['label']} modes: {list(data['modes'].keys())}")

## 3. Data Preparation & Summary Statistics

In [ ]:
def _stat(d, key, stat='mean', default=0):
    """Safely extract a statistic from a nested timing/quality dict."""
    return (d.get(key) or {}).get(stat, default) or default

def create_comparison_dataframe(results, dataset_name):
    """
    Create a fully enriched DataFrame from a comparison report.
    Extracts all timing sub-metrics, model quality metrics, communication
    bytes, memory, and FL configuration from each mode result.

    Total Time (s) is computed as genuine wall-clock time assuming clients
    run in parallel within each round:
        total_time = (client_fit_total + client_crypto_total) / num_clients
                     + server_aggregate_total
    This matches the formula used in report.py.
    """
    if not results or 'modes' not in results:
        return None

    rows = []
    for mode_name, mode_data in results['modes'].items():
        bm   = mode_data.get('benchmark', {}) or {}
        tim  = bm.get('timing', {}) or {}
        mem  = bm.get('memory_mb', {}) or {}
        comm = bm.get('communication_bytes', {}) or {}
        mq   = bm.get('model_quality', {}) or {}

        # -- FL config ---------------------------------------------------------
        num_clients = bm.get('num_clients', None) or 1
        rounds      = bm.get('rounds', None) or 1
        nc          = num_clients

        # -- Timing helpers ----------------------------------------------------
        def t(key): return _stat(tim, key, 'mean')    # mean per call (seconds)
        def tt(key): return _stat(tim, key, 'total')  # total across all calls

        # -- Identify active privacy operations --------------------------------
        has_he  = tt('encryption') + tt('decryption') > 0

        # -- Client-side crypto totals (summed across all nc x nr calls) -------
        client_enc    = tt('encryption') + tt('decryption')
        client_proof  = tt('proof_generation') + tt('proof_verification')
        client_dp     = tt('dp_noise_addition')
        client_crypto = client_enc + client_proof + client_dp

        # -- Server-side crypto -------------------------------------------------
        server_agg = tt('server_aggregate')
        server_crypto = server_agg if has_he else 0.0

        # -- Wall-clock totals --------------------------------------------------
        client_total = tt('client_fit') + client_crypto
        total_time   = client_total / nc + server_agg

        crypto         = client_crypto / nc + server_crypto
        crypto_per_rnd = crypto / rounds if rounds else 0.0
        training_time  = total_time - crypto

        # -- Communication ------------------------------------------------------
        upload_mean    = _stat(comm, 'upload', 'mean')
        download_mean  = _stat(comm, 'download', 'mean')
        upload_total   = _stat(comm, 'upload', 'total')
        download_total = _stat(comm, 'download', 'total')

        # -- Memory -------------------------------------------------------------
        client_mem_mean = _stat(mem, 'client_peak', 'mean')
        server_mem_mean = _stat(mem, 'server_peak', 'mean')
        client_mem_max  = _stat(mem, 'client_peak', 'max')
        server_mem_max  = _stat(mem, 'server_peak', 'max')

        # -- Model quality ------------------------------------------------------
        def q(key): return _stat(mq, key, 'mean')

        rows.append({
            # Identity
            'Mode': mode_name.upper().replace('_', '-'),
            'Dataset': dataset_name,
            'Clients': num_clients,
            'Rounds': rounds,
            'Success': bool(mode_data.get('success', True)),

            # Model quality
            'Train Acc (%)':   q('train_accuracy'),
            'Val Acc (%)':     q('val_accuracy'),
            'Test Acc (%)':    q('test_accuracy'),
            'Train Loss':      q('train_loss'),
            'Val Loss':        q('val_loss'),
            'Test Loss':       q('test_loss'),
            'Precision (%)':   q('test_precision'),
            'Recall (%)':      q('test_recall'),
            'F1 (%)':          q('test_f1'),
            'AUC-PRC (%)':     q('test_auprc'),
            'Threshold':       q('test_threshold'),

            # Timing (mean per round, seconds)
            'Fit Time (s)':       t('client_fit'),
            'Eval Time (s)':      t('client_eval'),
            'Get Params (s)':     t('client_get_params'),
            'Aggregate Time (s)': t('server_aggregate'),
            'Encrypt Time (s)':   t('encryption'),
            'Decrypt Time (s)':   t('decryption'),
            'Proof Gen (s)':      t('proof_generation'),
            'Proof Verify (s)':   t('proof_verification'),
            'DP Noise (s)':       t('dp_noise_addition'),

            # Timing totals
            'Total Fit (s)':       tt('client_fit'),
            'Total Encrypt (s)':   tt('encryption'),
            'Total Proof Gen (s)': tt('proof_generation'),
            'Crypto Overhead (s)': crypto,
            'Crypto Per Round (s)': crypto_per_rnd,
            'Training Time (s)':    training_time,
            'Total Time (s)':       total_time,

            # Communication
            'Upload Mean (KB)':     upload_mean / 1024,
            'Download Mean (KB)':   download_mean / 1024,
            'Upload Total (MB)':    upload_total / 1024 / 1024,
            'Download Total (MB)':  download_total / 1024 / 1024,

            # Memory
            'Client Mem Mean (MB)': client_mem_mean,
            'Server Mem Mean (MB)': server_mem_mean,
            'Client Mem Peak (MB)': client_mem_max,
            'Server Mem Peak (MB)': server_mem_max,
        })

    return pd.DataFrame(rows)

if 'DATASET_SPECS' not in globals():
    DATASET_SPECS = [
        {'run_key': 'healthcare', 'report_keys': ['healthcare'], 'label': 'Healthcare'},
        {'run_key': 'creditcard', 'report_keys': ['creditcard'], 'label': 'Credit Card'},
        {'run_key': 'stock', 'report_keys': ['stock'], 'label': 'Stock'},
        {'run_key': 'mnist', 'report_keys': ['mnist'], 'label': 'MNIST'},
        {'run_key': 'cifar10', 'report_keys': ['cifar10', 'cifar'], 'label': 'CIFAR-10'},
    ]

DATASET_ORDER = [s['run_key'] for s in DATASET_SPECS]
DATASET_LABELS = {s['run_key']: s['label'] for s in DATASET_SPECS}

df_by_key = {}
for key in DATASET_ORDER:
    raw = results_by_dataset.get(key) if 'results_by_dataset' in globals() else None
    label = DATASET_LABELS[key]
    df_by_key[key] = create_comparison_dataframe(raw, label)

# Keep backwards-compatible variables used in later cells
df_healthcare = df_by_key.get('healthcare')
df_creditcard = df_by_key.get('creditcard')
df_stock = df_by_key.get('stock')
df_mnist = df_by_key.get('mnist')
df_cifar10 = df_by_key.get('cifar10')
df_cifar = df_cifar10  # legacy alias

available = [df for df in df_by_key.values() if df is not None]
df_combined = pd.concat(available, ignore_index=True) if available else None

ALL_DATASETS = [
    (DATASET_LABELS[k], df_by_key[k])
    for k in DATASET_ORDER
    if df_by_key[k] is not None
]

print('\nLoaded datasets:')
for label, df in ALL_DATASETS:
    print(f"  {label}: {len(df)} modes")
    print(f"    Total Time range: {df['Total Time (s)'].min():.1f}s - {df['Total Time (s)'].max():.1f}s")
    print(f"    Crypto overhead range: {df['Crypto Overhead (s)'].min():.3f}s - {df['Crypto Overhead (s)'].max():.1f}s")

if df_combined is not None:
    print(f"\nTotal experiments: {len(df_combined)} | Success rate: {df_combined['Success'].mean()*100:.1f}%")

### 3.1 Display Summary Tables

In [ ]:
SUMMARY_COLS = [
    'Mode', 'Val Acc (%)', 'AUC-PRC (%)',
    'Fit Time (s)', 'Crypto Per Round (s)', 'Crypto Overhead (s)', 'Total Time (s)',
    'Upload Mean (KB)', 'Success'
]

def show_table(df, label, cols=SUMMARY_COLS):
    if df is None:
        print(f"⚠️  {label}: data not available")
        return
    available = [c for c in cols if c in df.columns]
    print("\n" + "="*120)
    print(f"📋 {label} RESULTS")
    print("="*120)
    display(df[available].round(3))

for label, df in ALL_DATASETS:
    show_table(df, f"{label.upper()} DATASET")


### 3.2 Detailed Timing Breakdown
All timing sub-metrics (mean seconds per round): fit, eval, aggregate, encryption/decryption, proof generation/verification, DP noise addition.

In [ ]:
TIMING_COLS = [
    'Mode',
    'Fit Time (s)', 'Eval Time (s)', 'Get Params (s)', 'Aggregate Time (s)',
    'Encrypt Time (s)', 'Decrypt Time (s)',
    'Proof Gen (s)', 'Proof Verify (s)',
    'DP Noise (s)',
    'Crypto Per Round (s)', 'Crypto Overhead (s)', 'Training Time (s)', 'Total Time (s)',
]

def show_timing(df, label):
    if df is None:
        print(f"⚠️  {label}: not available"); return
    cols = [c for c in TIMING_COLS if c in df.columns]
    print("\n" + "="*130)
    print(f"⏱  {label} — TIMING BREAKDOWN (mean per round · seconds)")
    print("="*130)
    styled = df[cols].round(4).style \
        .background_gradient(subset=[c for c in ['Crypto Overhead (s)', 'Total Time (s)'] if c in cols],
                             cmap='YlOrRd') \
        .format("{:.4f}", subset=[c for c in cols if c != 'Mode'])
    display(styled)

for label, df in ALL_DATASETS:
    show_timing(df, label)


### 3.3 Model Quality Deep Dive
Precision, Recall, F1, AUC-PRC, and loss metrics across all modes and datasets.

In [ ]:
MQ_COLS = [
    'Mode',
    'Train Acc (%)', 'Val Acc (%)', 'Test Acc (%)',
    'Precision (%)', 'Recall (%)', 'F1 (%)', 'AUC-PRC (%)',
    'Train Loss', 'Val Loss', 'Test Loss', 'Threshold',
]

def show_quality(df, label):
    if df is None:
        print(f"⚠️  {label}: not available"); return
    cols = [c for c in MQ_COLS if c in df.columns]
    print("\n" + "="*130)
    print(f"🎯 {label} — MODEL QUALITY METRICS (mean across rounds)")
    print("="*130)
    acc_cols  = [c for c in ['Val Acc (%)', 'Test Acc (%)', 'F1 (%)', 'AUC-PRC (%)', 'Precision (%)', 'Recall (%)'] if c in cols]
    loss_cols = [c for c in ['Train Loss', 'Val Loss', 'Test Loss'] if c in cols]
    styled = df[cols].round(4).style \
        .background_gradient(subset=acc_cols,  cmap='RdYlGn', vmin=50, vmax=100) \
        .background_gradient(subset=loss_cols, cmap='RdYlGn_r', vmin=0, vmax=1)
    display(styled)

for label, df in ALL_DATASETS:
    show_quality(df, label)

# ── Bar chart: Precision / Recall / F1 / AUC-PRC across datasets ─────────────
metric_cols = ['Precision (%)', 'Recall (%)', 'F1 (%)', 'AUC-PRC (%)']

if ALL_DATASETS:
    n_datasets = len(ALL_DATASETS)
    fig, axes = plt.subplots(1, n_datasets, figsize=(8 * n_datasets, 6), sharey=False)
    if n_datasets == 1:
        axes = [axes]

    for ax, (label, df) in zip(axes, ALL_DATASETS):
        avail = [c for c in metric_cols if c in df.columns and df[c].sum() > 0]
        if not avail:
            ax.set_title(f"{label}\n(no metrics)"); continue
        plot_df = df.set_index('Mode')[avail]
        plot_df.plot(kind='bar', ax=ax, rot=30, colormap='Set2', edgecolor='black', linewidth=0.5)
        ax.set_title(f"{label}", fontsize=13, fontweight='bold')
        ax.set_xlabel('Mode', fontsize=11)
        ax.set_ylabel('Score (%)', fontsize=11)
        ax.set_ylim(0, 110)
        ax.legend(loc='upper right', fontsize=9)
        ax.grid(axis='y', alpha=0.4)
        for bar in ax.patches:
            h = bar.get_height()
            if h > 0:
                ax.text(bar.get_x() + bar.get_width() / 2, h + 0.5,
                        f'{h:.1f}', ha='center', va='bottom', fontsize=7, rotation=90)

    plt.suptitle('Precision / Recall / F1 / AUC-PRC by Privacy Mode', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()


### 3.4 Resource Usage — Communication & Memory
Per-round communication sizes and peak memory consumption (client + server).

In [ ]:
RES_COLS = [
    'Mode', 'Clients', 'Rounds',
    'Upload Mean (KB)', 'Download Mean (KB)',
    'Upload Total (MB)', 'Download Total (MB)',
    'Client Mem Mean (MB)', 'Server Mem Mean (MB)',
    'Client Mem Peak (MB)', 'Server Mem Peak (MB)',
]

def show_resources(df, label):
    if df is None:
        print(f"⚠️  {label}: not available"); return
    cols = [c for c in RES_COLS if c in df.columns]
    print("\n" + "="*130)
    print(f"📡 {label} — RESOURCE USAGE (communication & memory)")
    print("="*130)
    comm_cols = [c for c in ['Upload Mean (KB)', 'Download Mean (KB)', 'Upload Total (MB)', 'Download Total (MB)'] if c in cols]
    mem_cols  = [c for c in ['Client Mem Mean (MB)', 'Server Mem Mean (MB)', 'Client Mem Peak (MB)', 'Server Mem Peak (MB)'] if c in cols]
    display(df[cols].round(3).style
            .background_gradient(subset=comm_cols, cmap='Blues')
            .background_gradient(subset=mem_cols,  cmap='Oranges'))

for label, df in ALL_DATASETS:
    show_resources(df, label)

# ── Memory bar chart ─────────────────────────────────────────────────────────
if ALL_DATASETS:
    fig, axes = plt.subplots(1, len(ALL_DATASETS), figsize=(7 * len(ALL_DATASETS), 5), sharey=False)
    if len(ALL_DATASETS) == 1:
        axes = [axes]
    for ax, (label, df) in zip(axes, ALL_DATASETS):
        mem_avail = [c for c in ['Client Mem Mean (MB)', 'Server Mem Mean (MB)'] if c in df.columns]
        if not mem_avail:
            continue
        plot_df = df.set_index('Mode')[mem_avail]
        plot_df.plot(kind='bar', ax=ax, rot=30, colormap='tab10', edgecolor='black', linewidth=0.4)
        ax.set_title(f"{label}", fontsize=12, fontweight='bold')
        ax.set_ylabel('Memory (MB)')
        ax.set_xlabel('Mode')
        ax.legend(fontsize=9)
        ax.grid(axis='y', alpha=0.4)
    plt.suptitle('Client & Server Peak Memory by Privacy Mode', fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

# ── Communication bar chart ───────────────────────────────────────────────────
if ALL_DATASETS:
    fig, axes = plt.subplots(1, len(ALL_DATASETS), figsize=(7 * len(ALL_DATASETS), 5), sharey=False)
    if len(ALL_DATASETS) == 1:
        axes = [axes]
    for ax, (label, df) in zip(axes, ALL_DATASETS):
        comm_avail = [c for c in ['Upload Mean (KB)', 'Download Mean (KB)'] if c in df.columns]
        if not comm_avail:
            continue
        plot_df = df.set_index('Mode')[comm_avail]
        plot_df.plot(kind='bar', ax=ax, rot=30, colormap='Set1', edgecolor='black', linewidth=0.4)
        ax.set_title(f"{label}", fontsize=12, fontweight='bold')
        ax.set_ylabel('Size (KB) per round')
        ax.set_xlabel('Mode')
        ax.legend(fontsize=9)
        ax.grid(axis='y', alpha=0.4)
    plt.suptitle('Upload / Download Per Round by Privacy Mode', fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()


### 3.5 Full Detail Table — All Metrics Combined
Every extracted field from the benchmark report in one sortable view.

In [ ]:
ALL_COLS_ORDER = [
    'Mode', 'Dataset', 'Clients', 'Rounds', 'Success',
    # accuracy
    'Train Acc (%)', 'Val Acc (%)', 'Test Acc (%)',
    'Precision (%)', 'Recall (%)', 'F1 (%)', 'AUC-PRC (%)',
    # loss
    'Train Loss', 'Val Loss', 'Test Loss', 'Threshold',
    # timing (mean per round)
    'Fit Time (s)', 'Eval Time (s)', 'Get Params (s)', 'Aggregate Time (s)',
    'Encrypt Time (s)', 'Decrypt Time (s)',
    'Proof Gen (s)', 'Proof Verify (s)', 'DP Noise (s)',
    'Crypto Per Round (s)', 'Crypto Overhead (s)', 'Training Time (s)', 'Total Time (s)',
    # communication
    'Upload Mean (KB)', 'Download Mean (KB)',
    'Upload Total (MB)', 'Download Total (MB)',
    # memory
    'Client Mem Mean (MB)', 'Server Mem Mean (MB)',
    'Client Mem Peak (MB)', 'Server Mem Peak (MB)',
]

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

for label, df in ALL_DATASETS:
    cols = [c for c in ALL_COLS_ORDER if c in df.columns]
    print("\n" + "="*150)
    print(f"📑 {label.upper()} — FULL DETAIL (all benchmark fields)")
    print("="*150)
    acc_cols  = [c for c in cols if '(%)' in c]
    loss_cols = [c for c in cols if 'Loss' in c]
    hi_cols   = [c for c in cols if c in ('Total Time (s)', 'Crypto Overhead (s)')]
    display(
        df[cols].round(4).style
        .background_gradient(subset=[c for c in acc_cols  if c in cols], cmap='RdYlGn', vmin=50, vmax=100)
        .background_gradient(subset=[c for c in loss_cols if c in cols], cmap='RdYlGn_r', vmin=0, vmax=1)
        .background_gradient(subset=[c for c in hi_cols   if c in cols], cmap='YlOrRd')
    )


## 4. Accuracy Comparison Visualizations

### 4.1 Accuracy Bar Chart (Side-by-Side)

In [ ]:
# Accuracy bar charts -- Train Acc and Val Acc, one subplot per available dataset
n = len(ALL_DATASETS)
if n == 0:
    print("No datasets available.")
else:
    titles = [lbl for lbl, _ in ALL_DATASETS]
    fig = make_subplots(rows=1, cols=n, subplot_titles=titles)

    # Dynamic colors so no dataset is dropped when n > 3
    base_colors = px.colors.qualitative.Safe
    train_colors = [base_colors[i % len(base_colors)] for i in range(n)]
    val_colors = [px.colors.qualitative.Pastel[i % len(px.colors.qualitative.Pastel)] for i in range(n)]

    for col_idx, (label, df) in enumerate(ALL_DATASETS, start=1):
        d = df.sort_values('Val Acc (%)', ascending=False)
        show_legend = (col_idx == 1)
        fig.add_trace(
            go.Bar(
                x=d['Mode'], y=d['Train Acc (%)'], name='Train Acc',
                marker_color=train_colors[col_idx - 1],
                text=d['Train Acc (%)'].round(1),
                textposition='outside', showlegend=show_legend
            ),
            row=1, col=col_idx
        )
        fig.add_trace(
            go.Bar(
                x=d['Mode'], y=d['Val Acc (%)'], name='Val Acc',
                marker_color=val_colors[col_idx - 1],
                text=d['Val Acc (%)'].round(1),
                textposition='outside', showlegend=show_legend
            ),
            row=1, col=col_idx
        )
        fig.update_yaxes(title_text='Accuracy (%)', range=[0, 110], row=1, col=col_idx)
        fig.update_xaxes(title_text='Privacy Mode', row=1, col=col_idx)

    fig.update_layout(
        title_text='<b>Training vs Validation Accuracy by Privacy Mode</b>',
        height=500, barmode='group', showlegend=True,
        template='plotly_white'
    )
    fig.show()

    # -- AUC-PRC bar chart (primary metric for imbalanced datasets) ------------
    if any('AUC-PRC (%)' in df.columns and df['AUC-PRC (%)'].sum() > 0
           for _, df in ALL_DATASETS):
        fig2 = make_subplots(rows=1, cols=n, subplot_titles=titles)
        prc_colors = [px.colors.qualitative.Bold[i % len(px.colors.qualitative.Bold)] for i in range(n)]

        for col_idx, (label, df) in enumerate(ALL_DATASETS, start=1):
            if 'AUC-PRC (%)' not in df.columns:
                continue
            d = df.sort_values('AUC-PRC (%)', ascending=False)
            fig2.add_trace(
                go.Bar(
                    x=d['Mode'], y=d['AUC-PRC (%)'], name=label,
                    marker_color=prc_colors[col_idx - 1],
                    text=d['AUC-PRC (%)'].round(2), textposition='outside',
                    showlegend=True
                ),
                row=1, col=col_idx
            )
            fig2.update_yaxes(title_text='AUC-PRC (%)', row=1, col=col_idx)
        fig2.update_layout(
            title_text='<b>AUC-PRC by Privacy Mode</b> (primary metric for imbalanced data)',
            height=450, template='plotly_white'
        )
        fig2.show()


### 4.2 Accuracy Heatmap

In [ ]:
# Create accuracy matrix for heatmap
accuracy_matrix = df_combined.pivot_table(
    index='Mode',
    columns='Dataset',
    values='Val Acc (%)',
    aggfunc='first'
)

# Create heatmap
fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(
    accuracy_matrix,
    annot=True,
    fmt='.1f',
    cmap='RdYlGn',
    center=70,
    vmin=50,
    vmax=90,
    cbar_kws={'label': 'Validation Accuracy (%)'},
    linewidths=0.5,
    ax=ax
)

plt.title('Validation Accuracy Heatmap Across Privacy Modes and Datasets',
          fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Dataset', fontsize=12, fontweight='bold')
plt.ylabel('Privacy Mode', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nKey Insights:")
for dataset_label in accuracy_matrix.columns:
    best_mode = accuracy_matrix[dataset_label].idxmax()
    best_val = accuracy_matrix[dataset_label].max()
    print(f"- {dataset_label} best mode: {best_mode} ({best_val:.1f}%)")

baseline_vals = df_combined[df_combined['Mode'] == 'BASELINE']['Val Acc (%)']
non_baseline_vals = df_combined[df_combined['Mode'] != 'BASELINE']['Val Acc (%)']
if not baseline_vals.empty and not non_baseline_vals.empty:
    avg_delta = baseline_vals.mean() - non_baseline_vals.mean()
    print(f"- Average accuracy loss vs baseline: {avg_delta:.2f}%")


## 5. Performance Analysis

### 5.1 Execution Time Breakdown

In [ ]:
# Stacked bar: training time vs cryptographic overhead (wall-clock, per run)
# Training Time = Total Time − Crypto Overhead (both correctly parallelism-adjusted)
# This shows how much of the total wall-clock is due to privacy operations.

n = len(ALL_DATASETS)
if n == 0:
    print("No datasets available.")
else:
    titles = [lbl for lbl, _ in ALL_DATASETS]
    fig = make_subplots(rows=1, cols=n, subplot_titles=titles)

    for col_idx, (label, df) in enumerate(ALL_DATASETS, start=1):
        d = df.copy()
        # Clamp to avoid negative training time if rounding
        d['Training Time (s)'] = d['Training Time (s)'].clip(lower=0)

        show_legend = (col_idx == 1)
        fig.add_trace(
            go.Bar(x=d['Mode'], y=d['Training Time (s)'],
                   name='Training Time', marker_color='steelblue',
                   showlegend=show_legend),
            row=1, col=col_idx
        )
        fig.add_trace(
            go.Bar(x=d['Mode'], y=d['Crypto Overhead (s)'],
                   name='Crypto Overhead', marker_color='orangered',
                   showlegend=show_legend),
            row=1, col=col_idx
        )
        fig.update_yaxes(title_text="Wall-clock time (s)", row=1, col=col_idx)
        fig.update_xaxes(title_text="Privacy Mode", row=1, col=col_idx)

    fig.update_layout(
        title_text='<b>Wall-clock Time: Training vs Cryptographic Overhead</b>',
        height=500,
        barmode='stack',
        template='plotly_white',
    )
    fig.show()

    # Print summary
    print("\n📊 Crypto overhead as % of total wall-clock time:")
    for label, df in ALL_DATASETS:
        print(f"\n  {label}:")
        for _, row in df.iterrows():
            pct = 100 * row['Crypto Overhead (s)'] / row['Total Time (s)'] if row['Total Time (s)'] > 0 else 0
            print(f"    {row['Mode']:<22}: {row['Total Time (s)']:6.1f}s total  |  {row['Crypto Overhead (s)']:6.2f}s crypto  ({pct:.1f}%)")


### 5.2 Overhead Ratio Analysis

In [ ]:
# Overhead ratio vs baseline for all datasets
fig = go.Figure()

palette = px.colors.qualitative.Dark24
dataset_colors = [palette[i % len(palette)] for i in range(len(ALL_DATASETS))]

dfs_with_overhead = []
for i, (label, df) in enumerate(ALL_DATASETS):
    baseline_rows = df[df['Mode'] == 'BASELINE']
    if baseline_rows.empty:
        print(f"⚠️  {label}: no baseline row, skipping overhead chart")
        continue
    baseline_time = baseline_rows['Total Time (s)'].values[0]
    d = df.copy()
    d['Overhead Ratio'] = d['Total Time (s)'] / baseline_time
    d['Overhead %'] = (d['Overhead Ratio'] - 1) * 100
    d['_label'] = label
    d['_color'] = dataset_colors[i]
    dfs_with_overhead.append((label, d, dataset_colors[i]))

    fig.add_trace(go.Bar(
        name=label,
        x=d['Mode'],
        y=d['Overhead Ratio'],
        text=d['Overhead Ratio'].round(2),
        textposition='outside',
        marker_color=dataset_colors[i]
    ))

fig.add_hline(y=1.0, line_dash='dash', line_color='red',
              annotation_text='Baseline (1.0x)', annotation_position='right')

fig.update_layout(
    title='<b>Performance Overhead Ratio vs Baseline</b>',
    xaxis_title='Privacy Mode',
    yaxis_title='Overhead Ratio (x)',
    barmode='group',
    height=500,
    template='plotly_white',
    showlegend=True
)
fig.show()

# Print overhead statistics
print("\nOverhead Statistics (wall-clock time):")
for label, d, _ in dfs_with_overhead:
    print(f"\n  {label}:")
    baseline_time = d[d['Mode'] == 'BASELINE']['Total Time (s)'].values[0]
    print(f"    Baseline: {baseline_time:.1f}s")
    for _, row in d[d['Mode'] != 'BASELINE'].iterrows():
        print(f"    {row['Mode']:<25}: {row['Overhead Ratio']:.2f}x ({row['Overhead %']:+.1f}%)")


### 5.3 Memory Usage Comparison

In [ ]:
# Memory usage comparison -- client vs server for all datasets
n = len(ALL_DATASETS)
if n == 0:
    print("No datasets available.")
else:
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Client Peak Memory', 'Server Peak Memory')
    )

    palette = px.colors.qualitative.Dark24
    dataset_colors = [palette[i % len(palette)] for i in range(n)]

    for i, (label, df) in enumerate(ALL_DATASETS):
        color = dataset_colors[i]
        fig.add_trace(
            go.Bar(
                x=df['Mode'], y=df['Client Mem Mean (MB)'],
                name=label, marker_color=color, opacity=0.75
            ),
            row=1, col=1
        )
        fig.add_trace(
            go.Bar(
                x=df['Mode'], y=df['Server Mem Mean (MB)'],
                name=label, marker_color=color, opacity=0.75, showlegend=False
            ),
            row=1, col=2
        )

    fig.update_layout(
        title_text='<b>Memory Usage Across Privacy Modes</b>',
        height=500, barmode='group', template='plotly_white'
    )
    fig.update_yaxes(title_text='Memory (MB)', row=1, col=1)
    fig.update_yaxes(title_text='Memory (MB)', row=1, col=2)
    fig.update_xaxes(title_text='Privacy Mode', row=1, col=1)
    fig.update_xaxes(title_text='Privacy Mode', row=1, col=2)
    fig.show()


## 6. Privacy-Performance Trade-off Analysis

### 6.1 Scatter Plot: Accuracy vs Performance

In [ ]:
# Create scatter plot for privacy-performance trade-off
fig = px.scatter(
    df_combined,
    x='Total Time (s)',
    y='Val Acc (%)',
    color='Mode',
    symbol='Dataset',
    size='Crypto Overhead (s)',
    hover_data=['Train Acc (%)', 'Fit Time (s)', 'Upload Mean (KB)'],
    title='<b>Privacy-Performance Trade-off: Accuracy vs Execution Time</b>',
    labels={
        'Total Time (s)': 'Total Execution Time (seconds)',
        'Val Acc (%)': 'Validation Accuracy (%)'
    },
    template='plotly_white',
    height=600,
    size_max=30
)

# Add quadrant lines
avg_time = df_combined['Total Time (s)'].mean()
avg_acc = df_combined['Val Acc (%)'].mean()

fig.add_vline(x=avg_time, line_dash="dash", line_color="gray", opacity=0.5)
fig.add_hline(y=avg_acc, line_dash="dash", line_color="gray", opacity=0.5)

# Add annotations for ideal region
fig.add_annotation(
    x=20, y=85,
    text="<b>IDEAL</b><br>(High Acc, Low Time)",
    showarrow=False,
    font=dict(size=12, color="green"),
    bgcolor="lightgreen",
    opacity=0.3
)

fig.update_layout(
    legend=dict(
        orientation="v",
        yanchor="top",
        y=0.99,
        xanchor="right",
        x=0.99
    )
)

fig.show()

### 6.2 Pareto Frontier Analysis

In [ ]:
def calculate_pareto_scores(df):
    """
    Calculate normalized privacy, accuracy, and performance scores.

    Privacy ordering (weakest -> strongest):
      baseline(0) < dp(4) < zkp/zkp_sampled(5) < he_tenseal(7) < he_concrete_tfhe(8)
      < he_tenseal_zkp(9) < he_concrete_tfhe_zkp(9.5) < he_tenseal_zkp_dp(9.75)
      < he_concrete_tfhe_zkp_dp(10)

    Rationale:
    - DP: formal epsilon-delta guarantee, but aggregator still sees noisy gradients
    - ZKP: integrity verification (no byzantine attacks), server still sees gradients
    - HE-TenSEAL: encrypted aggregation (confidentiality)
    - HE-Concrete-TFHE: exact FHE confidentiality with higher cost
    - Hybrids: HE confidentiality + ZKP integrity + optional DP guarantee
    """
    df = df.copy()

    PRIVACY_SCORES = {
        'BASELINE':                    0.0,
        'DP':                          4.0,
        'ZKP':                         5.0,
        'ZKP-SAMPLED':                 5.0,
        'HE-TENSEAL':                  7.0,
        'HE-CONCRETE-TFHE':            8.0,
        'HE-TENSEAL-ZKP':              9.0,
        'HE-CONCRETE-TFHE-ZKP':        9.5,
        'HE-TENSEAL-ZKP-DP':           9.75,
        'HE-CONCRETE-TFHE-ZKP-DP':     10.0,
    }
    df['Privacy Score'] = df['Mode'].map(PRIVACY_SCORES).fillna(0)

    # Primary accuracy metric: AUC-PRC if available and non-zero, else Val Acc
    use_auprc = 'AUC-PRC (%)' in df.columns and df['AUC-PRC (%)'].sum() > 0
    acc_col = 'AUC-PRC (%)' if use_auprc else 'Val Acc (%)'
    df['Accuracy Score'] = df[acc_col] / 100

    # Performance score (inverse of time overhead vs baseline)
    max_time = df['Total Time (s)'].max()
    df['Performance Score'] = 1 - (df['Total Time (s)'] / max_time)

    # Communication efficiency (inverse log scale -- HE upload is much larger)
    upload = df['Upload Mean (KB)'].replace(0, 0.001)
    df['Comm Score'] = 1 - (upload.apply(lambda x: np.log1p(x)) / np.log1p(upload.max() + 1))

    # Memory efficiency
    max_mem = df['Client Mem Mean (MB)'].max()
    df['Mem Score'] = 1 - (df['Client Mem Mean (MB)'] / max_mem) if max_mem > 0 else 1.0

    # Combined score (weighted)
    df['Combined Score'] = (
        0.35 * df['Accuracy Score'] +
        0.25 * df['Performance Score'] +
        0.25 * (df['Privacy Score'] / 10) +
        0.08 * df['Comm Score'] +
        0.07 * df['Mem Score']
    )
    return df, acc_col

# -- Pareto scores for all datasets -------------------------------------------
pareto_dfs = []
for label, df in ALL_DATASETS:
    scored, acc_col = calculate_pareto_scores(df)
    scored['_dataset'] = label
    pareto_dfs.append((label, scored, acc_col))

# -- Radar chart for the primary dataset (use first available) ----------------
if pareto_dfs:
    label0, df0, acc_col0 = pareto_dfs[0]
    fig = go.Figure()
    categories = ['Privacy', 'Accuracy', 'Performance', 'Comm Efficiency', 'Mem Efficiency']

    for _, row in df0.iterrows():
        values = [
            row['Privacy Score'] / 10,
            row['Accuracy Score'],
            row['Performance Score'],
            row['Comm Score'],
            row['Mem Score'],
        ]
        fig.add_trace(go.Scatterpolar(
            r=values + [values[0]],
            theta=categories + [categories[0]],
            name=row['Mode'],
            fill='toself',
            opacity=0.55
        ))

    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
        showlegend=True,
        title=f'<b>Multi-Dimensional Comparison ({label0})</b><br>'
              f'<sub>Accuracy={acc_col0} | Privacy scale: baseline=0, HE+ZKP+DP=10</sub>',
        height=600,
        template='plotly_white'
    )
    fig.show()

# -- Combined score table for all datasets ------------------------------------
for label, df, acc_col in pareto_dfs:
    top = df.nlargest(3, 'Combined Score')[
        ['Mode', 'Combined Score', 'Privacy Score', 'Accuracy Score', 'Performance Score', 'Comm Score']
    ]
    print(f"\nTop 3 Modes -- {label} (primary metric: {acc_col}):")
    display(top.round(3))


## 7. Comprehensive Comparison Table

In [ ]:
# Create comprehensive comparison table for all available datasets
COMP_COLS = [
    'Mode', 'Train Acc (%)', 'Val Acc (%)', 'F1 (%)', 'AUC-PRC (%)',
    'Fit Time (s)', 'Crypto Per Round (s)', 'Crypto Overhead (s)', 'Total Time (s)',
    'Upload Mean (KB)', 'Download Mean (KB)',
    'Client Mem Mean (MB)', 'Server Mem Mean (MB)'
]

for label, df in ALL_DATASETS:
    cols = [c for c in COMP_COLS if c in df.columns]
    numeric = df[cols].select_dtypes(include=[np.number]).columns.tolist()
    rounded = df[cols].copy()
    rounded[numeric] = rounded[numeric].round(3)

    # Gradient ranges depend on dataset characteristics
    acc_max = rounded['Val Acc (%)'].max() if 'Val Acc (%)' in cols else 100
    acc_min = max(acc_max - 10, 0)

    print("\n" + "="*130)
    print(f"📊 {label.upper()} — COMPREHENSIVE COMPARISON TABLE")
    print("="*130)
    styled = rounded.style
    if 'Val Acc (%)' in cols:
        styled = styled.background_gradient(subset=['Val Acc (%)'], cmap='RdYlGn',
                                            vmin=acc_min, vmax=acc_max)
    if 'AUC-PRC (%)' in cols and df['AUC-PRC (%)'].sum() > 0:
        auprc_max = rounded['AUC-PRC (%)'].max()
        styled = styled.background_gradient(subset=['AUC-PRC (%)'], cmap='Blues',
                                            vmin=auprc_max - 15, vmax=auprc_max)
    if 'Total Time (s)' in cols:
        styled = styled.background_gradient(subset=['Total Time (s)'], cmap='RdYlGn_r',
                                            vmin=0, vmax=df['Total Time (s)'].max())
    if 'Crypto Overhead (s)' in cols:
        styled = styled.background_gradient(subset=['Crypto Overhead (s)'], cmap='YlOrRd',
                                            vmin=0, vmax=df['Crypto Overhead (s)'].max())
    display(styled)


## 8. Statistical Analysis

In [ ]:
from IPython.display import display, Markdown, HTML
import pandas as pd

for label, df in ALL_DATASETS:
    use_auprc = 'AUC-PRC (%)' in df.columns and df['AUC-PRC (%)'].sum() > 0
    acc_col   = 'AUC-PRC (%)' if use_auprc else 'Val Acc (%)'
    acc_label = "AUC-PRC — primary metric (imbalanced data)" if use_auprc else "Validation Accuracy"

    baseline_row  = df[df['Mode'] == 'BASELINE']
    baseline_time = baseline_row['Total Time (s)'].values[0] if not baseline_row.empty else None
    baseline_acc  = baseline_row[acc_col].values[0]           if not baseline_row.empty else None

    best_mode  = df.loc[df[acc_col].idxmax(), 'Mode']
    worst_mode = df.loc[df[acc_col].idxmin(), 'Mode']

    # ── Accuracy delta table ──────────────────────────────────────────────────
    delta_rows = []
    for _, r in df.iterrows():
        delta = r[acc_col] - baseline_acc if baseline_acc is not None else 0
        ratio = r['Total Time (s)'] / baseline_time if baseline_time else 1.0
        up_kb = r['Upload Mean (KB)']
        up_str = f"{up_kb:.0f} KB" if up_kb < 1024 else f"{up_kb/1024:.1f} MB"
        delta_rows.append({
            'Mode': r['Mode'],
            acc_col: f"{r[acc_col]:.2f}%",
            'Δ Accuracy': "—" if r['Mode'] == 'BASELINE' else f"{delta:+.2f} pp",
            'Total Time (s)': f"{r['Total Time (s)']:.1f}",
            'Overhead': "—" if r['Mode'] == 'BASELINE' else f"×{ratio:.2f}",
            'Crypto / rnd': f"{r['Crypto Per Round (s)']:.3f} s" if r['Crypto Per Round (s)'] > 0.001 else "< 0.001 s",
            'Upload / rnd': up_str,
        })
    stat_df = pd.DataFrame(delta_rows).set_index('Mode')

    table_style = [
        {'selector': 'caption',  'props': [('font-weight', 'bold'), ('font-size', '1em'), ('text-align', 'left'), ('padding-bottom', '8px'), ('color', '#333')]},
        {'selector': 'th',       'props': [('background-color', '#2c3e50'), ('color', 'white'), ('padding', '8px 14px'), ('text-align', 'center')]},
        {'selector': 'td',       'props': [('padding', '6px 14px'), ('text-align', 'center'), ('border-bottom', '1px solid #eee')]},
        {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f8f9fa')]},
    ]

    display(Markdown(f"""
### 📊 {label}

**Accuracy metric:** {acc_label}  
Mean: **{df[acc_col].mean():.2f}%** (σ = {df[acc_col].std():.2f}pp) &nbsp;|&nbsp; Best: **{best_mode}** ({df[acc_col].max():.2f}%) &nbsp;|&nbsp; Worst: **{worst_mode}** ({df[acc_col].min():.2f}%)
"""))

    display(stat_df.style
        .set_caption(f"{label} — accuracy, overhead & communication per mode")
        .set_table_styles(table_style)
    )

# ── Overall summary ───────────────────────────────────────────────────────────
n_exp    = len(df_combined) if df_combined is not None else 0
n_ok     = int(df_combined['Success'].sum()) if df_combined is not None else 0
rate     = 100 * n_ok / n_exp if n_exp else 0
n_modes  = df_combined['Mode'].nunique() if df_combined is not None else 0
n_ds     = df_combined['Dataset'].nunique() if df_combined is not None else 0

display(Markdown(f"""
---
**Combined:** {n_exp} experiments · {n_modes} privacy modes · {n_ds} datasets · success rate **{rate:.1f}%**
"""))


## 9. Key Findings and Recommendations

In [ ]:
from IPython.display import display, Markdown
import pandas as pd

# -- Build findings dynamically from actual benchmark data --------------------
cc_label, cc_df = next(((l, d) for l, d in ALL_DATASETS if 'card' in l.lower()), ALL_DATASETS[0])
baseline_rows = cc_df[cc_df['Mode'] == 'BASELINE']
baseline_time = baseline_rows['Total Time (s)'].values[0] if not baseline_rows.empty else 1.0

# -- 1) Overhead table (all modes) --------------------------------------------
overhead_df = cc_df[['Mode', 'Total Time (s)', 'Crypto Per Round (s)', 'Upload Mean (KB)']].copy()
overhead_df['Overhead Ratio'] = overhead_df['Total Time (s)'] / baseline_time
overhead_df['Overhead'] = overhead_df['Overhead Ratio'].apply(lambda x: '—' if abs(x - 1.0) < 1e-9 else f'x{x:.2f}')
overhead_df['Upload / rnd'] = overhead_df['Upload Mean (KB)'].apply(
    lambda up: f"{up:.0f} KB" if up < 1024 else f"{up/1024:.1f} MB"
 )
overhead_df = overhead_df[['Mode', 'Total Time (s)', 'Overhead', 'Crypto Per Round (s)', 'Upload / rnd']]
overhead_df = overhead_df.sort_values('Total Time (s)').reset_index(drop=True)

# -- 2) Quality table (all modes) ---------------------------------------------
use_auprc = 'AUC-PRC (%)' in cc_df.columns and cc_df['AUC-PRC (%)'].sum() > 0
qual_col = 'AUC-PRC (%)' if use_auprc else 'Val Acc (%)'
baseline_qual = baseline_rows[qual_col].values[0] if not baseline_rows.empty else cc_df[qual_col].mean()

qual_df = cc_df[['Mode', qual_col]].copy()
qual_df['Delta vs Baseline (pp)'] = qual_df[qual_col] - baseline_qual
qual_df = qual_df.sort_values(qual_col, ascending=False).reset_index(drop=True)

# -- 3) Recommended use-case table (all modes) --------------------------------
def classify_mode(mode):
    m = mode.upper()
    if m == 'BASELINE':
        return 'No privacy baseline', 'Reference only', 'No privacy protections'
    if m == 'DP':
        return 'Client-side formal privacy', 'Mobile and edge clients', 'Accuracy and convergence tuning needed'
    if m in ('ZKP', 'ZKP-SAMPLED'):
        return 'Integrity and byzantine robustness', 'Auditability and poisoning defense', 'Proof generation overhead'
    if m == 'HE-TENSEAL':
        return 'Encrypted aggregation (efficient)', 'Cross-silo deployment with untrusted server', 'Approximate arithmetic semantics'
    if m == 'HE-CONCRETE-TFHE':
        return 'Encrypted aggregation (exact)', 'Exact FHE compliance scenarios', 'Large communication footprint'
    if m in ('HE-TENSEAL-ZKP', 'HE-CONCRETE-TFHE-ZKP'):
        return 'Confidentiality + integrity', 'High-assurance collaborative training', 'Combined HE + ZKP cost'
    if m in ('HE-TENSEAL-ZKP-DP', 'HE-CONCRETE-TFHE-ZKP-DP'):
        return 'Confidentiality + integrity + DP', 'Maximum defense-in-depth', 'Highest total overhead'
    return 'Custom', 'Review manually', 'Unknown mode'

reco_rows = []
for _, r in cc_df.iterrows():
    fit, use_case, constraint = classify_mode(r['Mode'])
    reco_rows.append({
        'Mode': r['Mode'],
        'Best Fit': fit,
        'Typical Use Case': use_case,
        'Main Constraint': constraint,
    })
reco_df = pd.DataFrame(reco_rows).sort_values('Mode').reset_index(drop=True)

display(Markdown(f"""
## 9. Key Findings and Recommendations

Primary reference dataset: **{cc_label}**
- Modes covered in this run: **{cc_df['Mode'].nunique()}**
- Rounds: **{int(cc_df['Rounds'].iloc[0])}** | Clients: **{int(cc_df['Clients'].iloc[0])}**
- Primary quality metric: **{qual_col}**
"""))

display(Markdown('### 1) Wall-clock and communication (all modes)'))
display(overhead_df.round({'Total Time (s)': 2, 'Crypto Per Round (s)': 3}))

display(Markdown('### 2) Model quality vs baseline (all modes)'))
display(qual_df.round({qual_col: 3, 'Delta vs Baseline (pp)': 3}))

display(Markdown('### 3) Practical recommendations (all modes)'))
display(reco_df)


## 10. Export Results

In [ ]:
# Export results to CSV for further analysis
output_dir = Path('results/comparison_analysis')
output_dir.mkdir(exist_ok=True)

# Save individual dataset tables
dataset_file_map = [
    ('healthcare', 'healthcare_detailed.csv'),
    ('creditcard', 'creditcard_detailed.csv'),
    ('stock', 'stock_detailed.csv'),
    ('mnist', 'mnist_detailed.csv'),
    ('cifar10', 'cifar10_detailed.csv'),
]

export_map = []
for key, fname in dataset_file_map:
    if 'df_by_key' in globals():
        export_map.append((fname, df_by_key.get(key)))
    else:
        export_map.append((fname, None))

# Legacy alias export for compatibility with older scripts
if 'df_cifar' in globals() and df_cifar is not None:
    export_map.append(('cifar_detailed.csv', df_cifar))

export_map.append(('combined_results.csv', df_combined))

for fname, df in export_map:
    if df is not None:
        df.to_csv(output_dir / fname, index=False)
        print(f"{fname}")
    else:
        print(f"{fname} -- dataset not available")

# Save pareto analysis if available
if pareto_dfs:
    for label, df, _ in pareto_dfs:
        slug = label.lower().replace(' ', '_').replace('-', '_')
        df.to_csv(output_dir / f'{slug}_pareto_scores.csv', index=False)
        print(f"{slug}_pareto_scores.csv")

# Summary statistics JSON
def safe_float(v):
    try:
        return float(v)
    except Exception:
        return 0.0

summary = {
    'total_experiments': len(df_combined) if df_combined is not None else 0,
    'datasets': {},
}
for label, df in ALL_DATASETS:
    key = label.lower().replace(' ', '_').replace('-', '_')
    use_auprc = 'AUC-PRC (%)' in df.columns and df['AUC-PRC (%)'].sum() > 0
    acc_col = 'AUC-PRC (%)' if use_auprc else 'Val Acc (%)'
    baseline_rows = df[df['Mode'] == 'BASELINE']
    fastest_privacy_rows = df[df['Mode'] != 'BASELINE'].nsmallest(1, 'Total Time (s)')
    summary['datasets'][key] = {
        'primary_metric': acc_col,
        'n_modes': len(df),
        'mean_accuracy': safe_float(df[acc_col].mean()),
        'std_accuracy': safe_float(df[acc_col].std()),
        'baseline_time_s': safe_float(baseline_rows['Total Time (s)'].values[0]) if not baseline_rows.empty else None,
        'best_mode': df.loc[df[acc_col].idxmax(), 'Mode'],
        'fastest_privacy_mode': fastest_privacy_rows['Mode'].values[0] if not fastest_privacy_rows.empty else None,
    }

with open(output_dir / 'summary_statistics.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('summary_statistics.json')
print(f"\nOutput directory: {output_dir.absolute()}")

## 11. Conclusion and Academic Discussion

This analysis benchmarks **all 10 privacy modes** across up to **5 datasets** (Healthcare, Credit Card, Stock, MNIST, CIFAR-10) in a unified FL pipeline.

The notebook now treats datasets and modes dynamically, so every section (tables, visualizations, Pareto scoring, and exports) automatically includes whatever datasets and modes are present in `results/<dataset>/comparison_report.json`.

---

### 11.1 Mode Coverage

Modes covered:
- Baseline
- DP
- ZKP
- ZKP-Sampled
- HE-TenSEAL
- HE-Concrete-TFHE
- HE-TenSEAL-ZKP
- HE-Concrete-TFHE-ZKP
- HE-TenSEAL-ZKP-DP
- HE-Concrete-TFHE-ZKP-DP

---

### 11.2 Practical Interpretation

- **DP** provides formal epsilon-delta privacy with comparatively low communication overhead.
- **ZKP / ZKP-Sampled** improves integrity and byzantine robustness, with proof-generation time as the main trade-off.
- **HE-TenSEAL** is the most communication-efficient encrypted aggregation mode for practical deployment.
- **HE-Concrete-TFHE** offers exact encrypted computation at significantly higher communication cost.
- **Hybrid modes** combine confidentiality (HE), integrity (ZKP), and optionally formal privacy (DP) for defense-in-depth at higher runtime cost.


---

### 11.3 Reproducibility Note

This notebook is configured to keep coverage complete by design:
- dataset loading is dynamic over Healthcare, Credit Card, Stock, MNIST, and CIFAR-10
- all 10 privacy modes are included in experiment command generation
- visualizations use dynamic color allocation so additional datasets are not dropped
- CSV export includes per-dataset outputs for every loaded dataset

Re-run all code cells after experiments complete to refresh figures and summary tables.

In [ ]:
# Coverage diagnostics: datasets, modes, and table column completeness
required_mode_set = {
    'BASELINE', 'DP', 'ZKP', 'ZKP-SAMPLED',
    'HE-TENSEAL', 'HE-CONCRETE-TFHE',
    'HE-TENSEAL-ZKP', 'HE-CONCRETE-TFHE-ZKP',
    'HE-TENSEAL-ZKP-DP', 'HE-CONCRETE-TFHE-ZKP-DP',
}

required_tables = {
    'SUMMARY_COLS': set(SUMMARY_COLS),
    'TIMING_COLS': set(TIMING_COLS),
    'MQ_COLS': set(MQ_COLS),
    'RES_COLS': set(RES_COLS),
    'ALL_COLS_ORDER': set(ALL_COLS_ORDER),
    'COMP_COLS': set(COMP_COLS),
}

print('=== DATASET COVERAGE ===')
print(f'Loaded datasets: {len(ALL_DATASETS)} -> {[label for label, _ in ALL_DATASETS]}')

print('\n=== MODE COVERAGE PER DATASET ===')
for label, df in ALL_DATASETS:
    modes = set(df['Mode'].astype(str).tolist())
    missing_modes = sorted(required_mode_set - modes)
    extra_modes = sorted(modes - required_mode_set)
    print(f'[{label}] modes={len(modes)}')
    print(f'  Missing required modes: {missing_modes if missing_modes else "None"}')
    print(f'  Extra/unexpected modes: {extra_modes if extra_modes else "None"}')

print('\n=== TABLE COLUMN COMPLETENESS ===')
for label, df in ALL_DATASETS:
    cols = set(df.columns)
    print(f'[{label}]')
    for tname, treq in required_tables.items():
        missing_cols = sorted(treq - cols)
        print(f'  {tname}: missing {len(missing_cols)} -> {missing_cols[:8]}{"..." if len(missing_cols) > 8 else ""}')

print('\n=== COMBINED SUMMARY ===')
if df_combined is not None:
    print(f'Total experiments: {len(df_combined)}')
    print(f'Unique datasets: {df_combined["Dataset"].nunique()}')
    print(f'Unique modes: {df_combined["Mode"].nunique()}')
    print(f'Global mode list: {sorted(df_combined["Mode"].unique().tolist())}')
else:
    print('df_combined is None')